# EfficientNet-B2 Signature Classification (Genuine vs. Forged)

This notebook provides a clean, self-contained workflow for signature forgery detection:
1. **Digital Image Processing (DIP)**: Binarization (Otsu), Contrast Stretching, Median Filter, High-Boost Filter, Sobel Gradient.
2. **Model Training**: Pretrained **EfficientNet-B2** binary classifier.
3. **Model Saving**: Checkpoint saved to `models/efficientnet_b2_signature.pt`.
4. **Export Results**: Training history, test predictions, and summary metrics saved to CSV in `results/`.
5. **Plots**: Training curves, Confusion Matrix, and ROC curve.

In [ ]:
import os
import glob
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import efficientnet_b2, EfficientNet_B2_Weights
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve, confusion_matrix, classification_report

# Setup compute device
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f'Using compute device: {device}')
os.makedirs('models', exist_ok=True)
os.makedirs('results', exist_ok=True)

## 1. Dataset Setup
Extracts any dataset zip file found or loads from `data/`.
Supports folders `data/train` and `data/test`, or automatically splits any genuine vs forgeries signature set.

In [ ]:
# Automatic dataset extraction if zip exists
zip_candidates = glob.glob('*.zip') + glob.glob('data/*.zip')
if zip_candidates and not (os.path.exists('data/train') and len(glob.glob('data/train/*/*')) > 0):
    print(f'Extracting {zip_candidates[0]} into data/ ...')
    with zipfile.ZipFile(zip_candidates[0], 'r') as z:
        z.extractall('data')
print('Dataset ready for training/testing.')

## 2. Digital Image Processing (DIP) Pipeline
Applies the sequential 4-stage DIP pipeline from the study.

In [ ]:
OUT_H, OUT_W = 192, 384
MEAN = np.array([0.485, 0.456, 0.406], np.float32).reshape(3, 1, 1)
STD = np.array([0.229, 0.224, 0.225], np.float32).reshape(3, 1, 1)

def ink_mask(gray, min_area=12):
    blur = cv2.GaussianBlur(gray, (3, 3), 0)
    _, mask = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    keep = np.zeros(n, bool)
    keep[1:] = stats[1:, cv2.CC_STAT_AREA] >= min_area
    return keep[labels]

def canonicalise(gray, binarize=True, margin=8):
    mask = ink_mask(gray)
    if not mask.any():
        return np.full_like(gray, 255)
    ys, xs = np.where(mask)
    y0, y1 = max(ys.min() - margin, 0), min(ys.max() + margin + 1, gray.shape[0])
    x0, x1 = max(xs.min() - margin, 0), min(xs.max() + margin + 1, gray.shape[1])
    g, m = gray[y0:y1, x0:x1], mask[y0:y1, x0:x1]
    out = np.full(g.shape, 255, np.uint8)
    if binarize:
        out[m] = 0
    else:
        ink = g[m].astype(np.float64)
        lo, hi = np.percentile(ink, 2), np.percentile(ink, 98)
        scaled = (ink - lo) / max(hi - lo, 1.0) * 160.0
        out[m] = np.clip(scaled, 0, 160).astype(np.uint8)
    return out

def letterbox(img, h=OUT_H, w=OUT_W, fill=255):
    s = min(h / img.shape[0], w / img.shape[1])
    nh, nw = max(int(round(img.shape[0] * s)), 1), max(int(round(img.shape[1] * s)), 1)
    r = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)
    out = np.full((h, w), fill, np.uint8)
    y, x = (h - nh) // 2, (w - nw) // 2
    out[y:y + nh, x:x + nw] = r
    return out

def contrast_stretching(img):
    r_min, r_max = float(np.min(img)), float(np.max(img))
    if r_max == r_min:
        return img.copy()
    stretched = (img.astype(np.float64) - r_min) * (255.0 / (r_max - r_min))
    return np.uint8(np.clip(stretched, 0, 255))

def median_filter(img, ksize=3):
    return cv2.medianBlur(img, ksize)

def high_boost_filter(img, ksize=3, k=1.5):
    blurred = cv2.blur(img, (ksize, ksize)).astype(np.float64)
    mask = img.astype(np.float64) - blurred
    boosted = img.astype(np.float64) + k * mask
    return np.uint8(np.clip(boosted, 0, 255))

def sobel_edge_detection(img, ksize=3):
    gx = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=ksize)
    gy = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=ksize)
    return np.uint8(np.clip(np.sqrt(gx**2 + gy**2), 0, 255))

def stages_to_channels(canon):
    s1 = contrast_stretching(canon)
    s2 = median_filter(s1, ksize=3)
    s3 = high_boost_filter(s2, ksize=3, k=1.5)
    s4 = sobel_edge_detection(s3, ksize=3)
    return np.stack([s2, s3, s4], axis=0)

def augment_signature(canon, rng):
    h, w = canon.shape
    ang, sc = rng.uniform(-6, 6), rng.uniform(0.82, 1.0)
    M = cv2.getRotationMatrix2D((w / 2, h / 2), ang, sc)
    M[0, 1] += np.tan(np.radians(rng.uniform(-8, 8)))
    M[:, 2] += (rng.uniform(-10, 10), rng.uniform(-8, 8))
    img = cv2.warpAffine(canon, M, (w, h), flags=cv2.INTER_LINEAR, borderValue=255)
    r = rng.random()
    if r < 0.25:
        img = cv2.erode(img, cv2.getStructuringElement(cv2.MORPH_CROSS, (3, 3)))
    elif r < 0.5:
        img = cv2.dilate(img, cv2.getStructuringElement(cv2.MORPH_CROSS, (3, 3)))
    return np.where(img < 128, 0, 255).astype(np.uint8)

## 3. Dataset Loader

In [ ]:
class SignatureDataset(Dataset):
    def __init__(self, file_paths, labels, is_train=False):
        self.file_paths = file_paths
        self.labels = np.array(labels, dtype=np.float32)
        self.is_train = is_train
        self.rng = np.random.default_rng()

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        path = self.file_paths[idx]
        gray = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if gray is None:
            gray = np.full((OUT_H, OUT_W), 255, dtype=np.uint8)
        canon = letterbox(canonicalise(gray, binarize=True))
        if self.is_train:
            canon = augment_signature(canon, self.rng)
        channels = stages_to_channels(canon)
        x = (channels.astype(np.float32) / 255.0 - MEAN) / STD
        return torch.from_numpy(x), torch.tensor(self.labels[idx])

def gather_data(data_root='data'):
    train_g = glob.glob(os.path.join(data_root, 'train', 'genuine', '*.*'))
    train_f = glob.glob(os.path.join(data_root, 'train', 'forged', '*.*'))
    test_g = glob.glob(os.path.join(data_root, 'test', 'genuine', '*.*'))
    test_f = glob.glob(os.path.join(data_root, 'test', 'forged', '*.*'))

    if train_g or train_f:
        tr_p = train_g + train_f
        tr_l = [0] * len(train_g) + [1] * len(train_f)
        te_p = test_g + test_f
        te_l = [0] * len(test_g) + [1] * len(test_f)
        return tr_p, tr_l, te_p, te_l

    all_files = glob.glob(os.path.join(data_root, '**', '*.*'), recursive=True)
    img_exts = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}
    paths, labels = [], []
    for f in all_files:
        ext = os.path.splitext(f)[1].lower()
        if ext in img_exts:
            fn = os.path.basename(f).lower()
            parent = os.path.basename(os.path.dirname(f)).lower()
            if 'original' in fn or 'genuine' in fn or 'original' in parent or 'genuine' in parent:
                paths.append(f)
                labels.append(0)
            elif 'forg' in fn or 'forg' in parent:
                paths.append(f)
                labels.append(1)

    if len(paths) > 0:
        from sklearn.model_selection import train_test_split
        return train_test_split(paths, labels, test_size=0.2, random_state=42, stratify=labels)

    return [], [], [], []

train_paths, test_paths, train_labels, test_labels = gather_data('data')
print(f'Training signatures: {len(train_paths)}, Test signatures: {len(test_paths)}')

## 4. EfficientNet-B2 Model Architecture
We load `efficientnet_b2` with a single linear classification head (BCE logit for forged probability).

In [ ]:
def build_model(pretrained=True, dropout=0.5):
    weights = EfficientNet_B2_Weights.DEFAULT if pretrained else None
    m = efficientnet_b2(weights=weights)
    m.classifier[0].p = dropout
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, 1)
    return m

model = build_model(pretrained=True).to(device)
print('EfficientNet-B2 instantiated successfully.')

## 5. Training & Checkpoint Saving
Runs model training, tracks accuracy and AUC on validation images, and saves best model to `models/efficientnet_b2_signature.pt`.

In [ ]:
def train(model, train_paths, train_labels, test_paths, test_labels, epochs=10, batch_size=16, lr=1e-4):
    if len(train_paths) == 0:
        print('No images found in data/ yet. Place signatures in data/ or use pre-trained checkpoint.')
        return [], 0.5

    train_ds = SignatureDataset(train_paths, train_labels, is_train=True)
    test_ds = SignatureDataset(test_paths, test_labels, is_train=False)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

    head = list(model.classifier.parameters())
    head_ids = {id(p) for p in head}
    body = [p for p in model.parameters() if id(p) not in head_ids]
    optimizer = torch.optim.AdamW([{'params': body, 'lr': lr}, {'params': head, 'lr': lr * 10}], weight_decay=0.05)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.BCEWithLogitsLoss()

    best_auc = -1.0
    best_thr = 0.5
    history = []

    for ep in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            smoothed_y = yb * 0.9 + 0.05
            optimizer.zero_grad()
            out = model(xb).squeeze(1)
            loss = criterion(out, smoothed_y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(yb)

        scheduler.step()
        running_loss /= len(train_ds)

        # Evaluation
        model.eval()
        v_logits, v_targets = [], []
        with torch.no_grad():
            for xb, yb in test_loader:
                xb = xb.to(device)
                v_logits.append(model(xb).squeeze(1).cpu().numpy())
                v_targets.append(yb.numpy())

        if len(v_logits) > 0:
            logits = np.concatenate(v_logits)
            targets = np.concatenate(v_targets)
            probs = 1.0 / (1.0 + np.exp(-logits))
            acc = accuracy_score(targets, (probs >= 0.5).astype(int))
            auc = roc_auc_score(targets, probs) if len(np.unique(targets)) > 1 else 0.5
            print(f'Epoch [{ep:02d}/{epochs:02d}] Loss: {running_loss:.4f} | Val Acc: {acc:.4f} | Val AUC: {auc:.4f}')
            history.append({'epoch': ep, 'train_loss': running_loss, 'val_acc': acc, 'val_auc': auc})

            if auc > best_auc:
                best_auc = auc
                fpr, tpr, thresholds = roc_curve(targets, probs)
                fnr = 1 - tpr
                eer_idx = np.nanargmin(np.abs(fnr - fpr))
                best_thr = float(thresholds[eer_idx])

                torch.save({
                    'model': model.state_dict(),
                    'thr': float(logits[eer_idx]) if eer_idx < len(logits) else 0.0,
                    'prob_thr': best_thr,
                    'val_auc': best_auc
                }, 'models/efficientnet_b2_signature.pt')
                print('  --> Saved models/efficientnet_b2_signature.pt')

    if history:
        pd.DataFrame(history).to_csv('results/training_history.csv', index=False)
        print('Saved results/training_history.csv')
    return history, best_thr

# Start training
history, calibrated_thr = train(model, train_paths, train_labels, test_paths, test_labels, epochs=10)

## 6. Testing, Evaluation, and Exporting Results
Evaluates test images, exports predictions and metrics to CSV.

In [ ]:
def evaluate(model, test_paths, test_labels, thr=0.5):
    if len(test_paths) == 0:
        print('No test images available.')
        return None, None, None, None

    ckpt_path = 'models/efficientnet_b2_signature.pt'
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ckpt['model'])
        thr = ckpt.get('prob_thr', thr)

    model.eval()
    test_ds = SignatureDataset(test_paths, test_labels, is_train=False)
    loader = DataLoader(test_ds, batch_size=16, shuffle=False)

    all_logits, all_targets = [], []
    with torch.no_grad():
        for xb, yb in loader:
            all_logits.append(model(xb.to(device)).squeeze(1).cpu().numpy())
            all_targets.append(yb.numpy())

    logits = np.concatenate(all_logits)
    targets = np.concatenate(all_targets).astype(int)
    probs = 1.0 / (1.0 + np.exp(-logits))
    preds = (probs >= thr).astype(int)

    # 1. Predictions CSV
    pred_df = pd.DataFrame({
        'image_path': test_paths,
        'true_label': targets,
        'true_class': ['Forged' if t == 1 else 'Genuine' for t in targets],
        'forged_prob': probs,
        'predicted_label': preds,
        'predicted_class': ['Forged' if p == 1 else 'Genuine' for p in preds],
        'is_correct': preds == targets
    })
    pred_df.to_csv('results/test_predictions.csv', index=False)
    print('Saved results/test_predictions.csv')

    # 2. Metrics CSV
    acc = accuracy_score(targets, preds)
    auc = roc_auc_score(targets, probs) if len(np.unique(targets)) > 1 else 0.5
    cm = confusion_matrix(targets, preds)
    tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)
    far = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    frr = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    metrics_df = pd.DataFrame([{
        'Accuracy': acc,
        'AUC': auc,
        'Threshold': thr,
        'FAR': far,
        'FRR': frr,
        'True Positives (Forged)': tp,
        'True Negatives (Genuine)': tn,
        'False Positives (Gen->Forged)': fp,
        'False Negatives (Forged->Gen)': fn
    }])
    metrics_df.to_csv('results/evaluation_metrics.csv', index=False)
    print('Saved results/evaluation_metrics.csv')
    display(metrics_df)
    return pred_df, metrics_df, probs, targets

pred_df, metrics_df, test_probs, test_targets = evaluate(model, test_paths, test_labels, thr=calibrated_thr)

## 7. Plots and Visualizations
Plots training loss, validation metrics, confusion matrix, and ROC curve.

In [ ]:
# Plot 1: Training Loss and Validation Metrics
if os.path.exists('results/training_history.csv'):
    hist_df = pd.read_csv('results/training_history.csv')
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(hist_df['epoch'], hist_df['train_loss'], 'o-', color='#DC2626', label='Train Loss')
    axes[0].set_title('Training Loss', fontsize=13, fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].grid(True, linestyle='--', alpha=0.6)
    axes[0].legend()

    axes[1].plot(hist_df['epoch'], hist_df['val_acc'], 's-', color='#2563EB', label='Val Accuracy')
    axes[1].plot(hist_df['epoch'], hist_df['val_auc'], '^-', color='#059669', label='Val AUC')
    axes[1].set_title('Validation Performance', fontsize=13, fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Score')
    axes[1].grid(True, linestyle='--', alpha=0.6)
    axes[1].legend()

    plt.tight_layout()
    plt.savefig('results/training_curves.png', dpi=300)
    plt.show()

# Plot 2: Confusion Matrix & ROC Curve
if pred_df is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    cm = confusion_matrix(test_targets, pred_df['predicted_label'].values)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[0],
                xticklabels=['Genuine', 'Forged'], yticklabels=['Genuine', 'Forged'])
    axes[0].set_title('Confusion Matrix', fontsize=13, fontweight='bold')
    axes[0].set_xlabel('Predicted Label')
    axes[0].set_ylabel('True Label')

    fpr, tpr, _ = roc_curve(test_targets, test_probs)
    auc_val = roc_auc_score(test_targets, test_probs)
    axes[1].plot(fpr, tpr, color='#7C3AED', lw=2, label=f'ROC (AUC = {auc_val:.3f})')
    axes[1].plot([0, 1], [0, 1], color='gray', linestyle='--')
    axes[1].set_title('ROC Curve', fontsize=13, fontweight='bold')
    axes[1].set_xlabel('False Positive Rate (FAR)')
    axes[1].set_ylabel('True Positive Rate (1 - FRR)')
    axes[1].grid(True, linestyle='--', alpha=0.6)
    axes[1].legend()

    plt.tight_layout()
    plt.savefig('results/evaluation_plots.png', dpi=300)
    plt.show()